# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. This package enables easy access to datasets described by the Croissant metadata schema, making it simple to audit record sets and fields and perform exploratory data analysis.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and prepare for record exploration using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the FAIR² dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers. Referencing record sets, fields, and columns by `@id` ensures reproducibility when processing and selecting specific data elements.

In [ ]:
# List all record sets and their fields with @id
record_sets = dataset.metadata.record_sets

for rs in record_sets:
    print(f"Record Set: {rs.name} | @id: {rs.id}")
    print("  Description:", getattr(rs, 'description', ''))
    print("  Fields (@id):")
    for fld in rs.fields:
        print(f"    - {fld.name} (@id: {fld.id}) - type: {getattr(fld, 'data_type', '')}")
    print('-' * 60)

## 3. Data Extraction
Load data from each available record set. Use the record set and field `@id`s from the overview above.

Below, we load all major tabular record sets into pandas DataFrames for further analysis.

In [ ]:
# Collect all record set @id fields
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records_iter = dataset.records(record_set=rs_id)
    records = list(records_iter)
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set '{rs_id}'. Columns: {df.columns.tolist()}")
    print(df.head(), "\n---\n")

## 4. Exploratory Data Analysis (EDA)
Let's explore and preprocess the main quantitative record set. We'll:
- Filter records based on a numeric field (e.g., `cr:peptide_count` or `cr:abundance`)
- Normalize and summarize key numeric fields
- Group by a categorical variable if available.

Pick the main record set and relevant fields using their `@id` (as listed previously) for robust referencing.

In [ ]:
# Example: Choose main protein quantification record set by @id and select numeric/categorical fields (update as needed)
# NOTE: Update <protein_records_id> and field ids if IDs differ in your schema.
protein_records_id = record_set_ids[0]  # Use the first as an example; edit if dataset structure changes
df = dataframes[protein_records_id]

# Try to identify common quantitative fields
numeric_candidates = [c for c in df.columns if ('abundance' in c.lower() or 'count' in c.lower() or 'mw' in c.lower() or 'coverage' in c.lower()) and pd.api.types.is_numeric_dtype(df[c])]
if len(numeric_candidates) == 0:
    # Try fallback to float/integer columns
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
if not numeric_candidates:
    print("No obvious numeric field found; update this cell with a valid numeric field @id.")
else:
    numeric_field = numeric_candidates[0]
    print(f"Selected numeric field: {numeric_field}")

    # Filter and normalize field
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered to {len(filtered_df)} records with {numeric_field} > {threshold}.")
    
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a categorical grouping field
    non_numeric = [c for c in df.columns if c not in numeric_candidates and df[c].nunique() < min(5, len(df)//2)]
    group_field = non_numeric[0] if non_numeric else None
    if group_field is not None:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by '{group_field}':\n", grouped)

## 5. Visualization
Visualize the distribution of the selected numeric field after filtering and normalization, and (optionally) by groups if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field in filtered_df:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.show()

    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, color='orange')
    plt.title(f"Distribution of Normalized {numeric_field} (filtered)")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.show()
    
    if group_field is not None:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.show()


## 6. Conclusion
- We demonstrated how to load and explore the FAIR² dataset using the `mlcroissant` library.
- Data was accessed by proper referencing of `@id` in record sets and fields to ensure robust and reproducible selections.
- We performed initial numeric filtering, normalization, and grouping for exploratory analysis.
- This approach can be extended to other fields and more advanced preprocessing as required for downstream analytics or machine learning.
